# 1. MMSE preprocessing

In this notebook, preprocess the complete longitudinal ADNI MMSE table while preserving the raw source unchanged.

make each decision only after inspecting the relevant output. The workflow is organised so that:

- raw values are preserved before cleaning;
- structural missingness is distinguished from true missing data;
- phase-specific MMSE fields are harmonised carefully;
- unusable assessments cannot contribute modelling domain scores;
- quality-control variables remain separate from model-ready features;
- potentially duplicated or inconsistent records are reviewed rather than silently removed;
- no missing values are imputed;
- no reference-date or baseline-window selection is performed here.

This corrected notebook writes versioned `_v2` outputs and therefore does not overwrite the files created by the earlier MMSE notebook.

## 1.1. Connect Google Drive and define the project paths

connect Google Drive and define the same non-imaging project structure used in the other preprocessing notebooks.

The raw MMSE and data-dictionary files will only be read. All outputs from this notebook will use `_v2` filenames so that the previous results remain available for comparison.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/adni_mri/adni_non_imaging")

RAW_DIR = PROJECT_DIR / "raw"
INVENTORY_DIR = PROJECT_DIR / "inventory"
INTERIM_DIR = PROJECT_DIR / "interim"
PROCESSED_DIR = PROJECT_DIR / "processed"
MANIFESTS_DIR = PROJECT_DIR / "manifests"
QC_DIR = PROJECT_DIR / "qc"

COGNITIVE_FUNCTIONAL_DIR = (
    RAW_DIR / "Cognitive and functional assessments"
)

SOURCE_TABLE_DIR = (
    RAW_DIR / "Cohort, dates and source-of-truth tables"
)

MMSE_PATH = (
    COGNITIVE_FUNCTIONAL_DIR
    / "All_Subjects_MMSE_11Jul2026.csv"
)

DATADIC_PATH = (
    SOURCE_TABLE_DIR
    / "DATADIC_11Jul2026.csv"
)

MMSE_INTERIM_DIR = INTERIM_DIR / "mmse"
MMSE_PROCESSED_DIR = PROCESSED_DIR / "mmse"
MMSE_QC_DIR = QC_DIR / "mmse"

for folder in [
    MMSE_INTERIM_DIR,
    MMSE_PROCESSED_DIR,
    MMSE_QC_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

required_paths = {
    "MMSE": MMSE_PATH,
    "DATADIC": DATADIC_PATH,
}

missing_paths = [
    str(path)
    for path in required_paths.values()
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_paths)
    )

print("Google Drive connected successfully.")
print(f"Project directory: {PROJECT_DIR}")
print(f"MMSE source: {MMSE_PATH}")
print(f"DATADIC source: {DATADIC_PATH}")

## 1.2. Load the raw MMSE table

load the full MMSE export and inspect its dimensions, participant coverage, phases, column names, and representative rows before making any changes.

In [ ]:
import numpy as np
import pandas as pd

mmse_raw = pd.read_csv(
    MMSE_PATH,
    low_memory=False,
)

print("MMSE table loaded successfully.")
print(f"Rows: {len(mmse_raw):,}")
print(f"Columns: {mmse_raw.shape[1]:,}")
print(
    "Unique participants: "
    f"{mmse_raw['RID'].nunique(dropna=True):,}"
)

print("\nRows by ADNI phase:")
display(
    mmse_raw["PHASE"]
    .value_counts(dropna=False)
    .rename_axis("PHASE")
    .reset_index(name="ROW_COUNT")
)

print("\nColumn names:")
for number, column in enumerate(mmse_raw.columns, start=1):
    print(f"{number:>3}. {column}")

display(mmse_raw.head())

## 1.3. Inspect raw data types, missingness, and representative values

create a column-level audit showing data type, missingness, number of unique values, and representative entries. This provides the baseline against which later transformations can be checked.

In [ ]:
column_inspection = pd.DataFrame(
    {
        "COLUMN_NAME": mmse_raw.columns,
        "DATA_TYPE": [
            str(mmse_raw[column].dtype)
            for column in mmse_raw.columns
        ],
        "NON_MISSING": [
            int(mmse_raw[column].notna().sum())
            for column in mmse_raw.columns
        ],
        "MISSING": [
            int(mmse_raw[column].isna().sum())
            for column in mmse_raw.columns
        ],
        "MISSING_PERCENT": [
            round(mmse_raw[column].isna().mean() * 100, 2)
            for column in mmse_raw.columns
        ],
        "UNIQUE_VALUES": [
            int(mmse_raw[column].nunique(dropna=True))
            for column in mmse_raw.columns
        ],
        "SAMPLE_ENTRIES": [
            mmse_raw[column]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(5)
            .tolist()
            for column in mmse_raw.columns
        ],
    }
)

display(column_inspection)

## 1.4. Load and restrict the ADNI data dictionary to the MMSE table

use `DATADIC` to interpret the MMSE variables. Because common fields such as `RID`, `VISCODE`, and `ID` occur in many ADNI tables, restrict the dictionary to `TBLNAME = MMSE` before matching field names.

In [ ]:
datadic_raw = pd.read_csv(
    DATADIC_PATH,
    low_memory=False,
)

mmse_datadic = datadic_raw.loc[
    datadic_raw["TBLNAME"]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq("MMSE")
].copy()

mmse_datadic["_FIELD_NORMALISED"] = (
    mmse_datadic["FLDNAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

export_columns = {
    str(column).strip().upper()
    for column in mmse_raw.columns
}

mmse_datadic_relevant = mmse_datadic.loc[
    mmse_datadic["_FIELD_NORMALISED"].isin(export_columns)
].copy()

mmse_datadic_relevant.insert(
    0,
    "MMSE_COLUMN",
    mmse_datadic_relevant["_FIELD_NORMALISED"],
)

mmse_datadic_relevant = (
    mmse_datadic_relevant
    .drop(columns="_FIELD_NORMALISED")
    .sort_values(
        ["MMSE_COLUMN", "PHASE", "DD_CRF_VERSION"],
        kind="stable",
    )
    .reset_index(drop=True)
)

match_counts = (
    mmse_datadic_relevant
    .groupby("MMSE_COLUMN")
    .size()
    .rename("MMSE_DATADIC_MATCH_COUNT")
)

dictionary_coverage = column_inspection.copy()
dictionary_coverage["MMSE_DATADIC_MATCH_COUNT"] = (
    dictionary_coverage["COLUMN_NAME"]
    .str.upper()
    .map(match_counts)
    .fillna(0)
    .astype(int)
)
dictionary_coverage["FOUND_IN_MMSE_DATADIC"] = (
    dictionary_coverage["MMSE_DATADIC_MATCH_COUNT"] > 0
)

print(
    "MMSE export columns documented in the MMSE dictionary: "
    f"{dictionary_coverage['FOUND_IN_MMSE_DATADIC'].sum()} "
    f"/ {len(dictionary_coverage)}"
)

display(dictionary_coverage)

## 1.5. Create a readable MMSE variable-definition summary

combine distinct phase-specific dictionary definitions into one readable row per exported variable. This table will preserve differences in descriptions, ranges, codes, and mapping notes.

In [ ]:
def combine_unique_text(series):
    values = (
        series.dropna()
        .astype(str)
        .str.strip()
    )
    values = values.loc[
        values.ne("")
        & values.str.lower().ne("nan")
    ]
    return " | ".join(dict.fromkeys(values))


mmse_definition_summary = (
    mmse_datadic_relevant
    .groupby("MMSE_COLUMN", sort=False, as_index=False)
    .agg(
        PHASES=("PHASE", combine_unique_text),
        DESCRIPTION=("TEXT", combine_unique_text),
        TYPE=("TYPE", combine_unique_text),
        VALID_RANGE_OR_LENGTH=("LENGTH", combine_unique_text),
        CODES=("CODE", combine_unique_text),
        UNITS=("UNITS", combine_unique_text),
        STATUS=("STATUS", combine_unique_text),
        CODE_CHANGES=("CODE_CHANGES", combine_unique_text),
        MAPPING_NOTES=("MAPPING_NOTES", combine_unique_text),
    )
)

mmse_definition_summary = (
    pd.DataFrame({"MMSE_COLUMN": mmse_raw.columns})
    .merge(
        mmse_definition_summary,
        on="MMSE_COLUMN",
        how="left",
    )
    .merge(
        column_inspection.rename(
            columns={"COLUMN_NAME": "MMSE_COLUMN"}
        ),
        on="MMSE_COLUMN",
        how="left",
    )
)

display(mmse_definition_summary)

## 1.6. Review phase-specific field availability

inspect missingness by ADNI phase for the fields whose structure changed across phases. This is necessary to distinguish structural absence from ordinary missing values.

In [ ]:
phase_structure_columns = [
    "DONE",
    "NDREASON",
    "SOURCE",
    "WORDLIST",
    "WORD1",
    "WORD2",
    "WORD3",
    "MMTRIALS",
    "MMD",
    "MML",
    "MMR",
    "MMO",
    "MMW",
    "MMLTR1",
    "MMLTR2",
    "MMLTR3",
    "MMLTR4",
    "MMLTR5",
    "MMLTR6",
    "MMLTR7",
    "WORLDSCORE",
    "WORD1DL",
    "WORD2DL",
    "WORD3DL",
    "MMSCORE",
    "LANGUAGE_CODE",
    "HAS_QC_ERROR",
]

phase_missingness = []

for phase, phase_data in mmse_raw.groupby("PHASE", dropna=False):
    for column in phase_structure_columns:
        phase_missingness.append(
            {
                "PHASE": phase,
                "COLUMN_NAME": column,
                "ROW_COUNT": len(phase_data),
                "NON_MISSING": int(
                    phase_data[column].notna().sum()
                ),
                "MISSING_PERCENT": round(
                    phase_data[column].isna().mean() * 100,
                    2,
                ),
            }
        )

phase_missingness = pd.DataFrame(phase_missingness)

display(
    phase_missingness.pivot(
        index="COLUMN_NAME",
        columns="PHASE",
        values="MISSING_PERCENT",
    )
)

## 1.7. Review special codes and valid ranges before cleaning

inspect the observed values of the completion fields, total score, trial count, attention score, word-list indicator, language, and QC flag.

also verify that all binary item scores remain within 0 and 1. No values will be changed in this step.

In [ ]:
audit_variables = [
    "DONE",
    "NDREASON",
    "SOURCE",
    "MMTRIALS",
    "MMSCORE",
    "WORDLIST",
    "WORLDSCORE",
    "HAS_QC_ERROR",
    "LANGUAGE_CODE",
]

for variable in audit_variables:
    print("\n" + "=" * 80)
    print(variable)
    print("=" * 80)
    display(
        mmse_raw[variable]
        .value_counts(dropna=False)
        .rename_axis(variable)
        .reset_index(name="ROW_COUNT")
    )

binary_item_columns = [
    "MMDATE", "MMYEAR", "MMMONTH", "MMDAY", "MMSEASON",
    "MMHOSPIT", "MMFLOOR", "MMCITY", "MMAREA", "MMSTATE",
    "WORD1", "WORD2", "WORD3",
    "MMD", "MML", "MMR", "MMO", "MMW",
    "WORD1DL", "WORD2DL", "WORD3DL",
    "MMWATCH", "MMPENCIL", "MMREPEAT",
    "MMHAND", "MMFOLD", "MMONFLR",
    "MMREAD", "MMWRITE", "MMDRAW",
]

invalid_binary_summary = []

for column in binary_item_columns:
    invalid_mask = (
        mmse_raw[column].notna()
        & ~mmse_raw[column].isin([0, 1])
    )
    invalid_binary_summary.append(
        {
            "COLUMN_NAME": column,
            "INVALID_ROW_COUNT": int(invalid_mask.sum()),
            "INVALID_VALUES": sorted(
                mmse_raw.loc[invalid_mask, column]
                .dropna()
                .unique()
                .tolist()
            ),
        }
    )

invalid_binary_summary = pd.DataFrame(
    invalid_binary_summary
)

print("\nBinary item range audit:")
display(invalid_binary_summary)

invalid_mmscore_rows = mmse_raw.loc[
    mmse_raw["MMSCORE"].notna()
    & ~mmse_raw["MMSCORE"].between(0, 30),
    [
        "PHASE", "PTID", "RID",
        "VISCODE", "VISCODE2", "VISDATE",
        "MMSCORE",
    ],
]

invalid_mmtrials_rows = mmse_raw.loc[
    mmse_raw["MMTRIALS"].notna()
    & ~mmse_raw["MMTRIALS"].between(1, 6),
    [
        "PHASE", "PTID", "RID",
        "VISCODE", "VISCODE2", "VISDATE",
        "MMTRIALS", "MMSCORE",
    ],
]

print(
    "\nMMSCORE values outside 0–30: "
    f"{len(invalid_mmscore_rows):,}"
)
display(invalid_mmscore_rows)

print(
    "\nMMTRIALS values outside 1–6: "
    f"{len(invalid_mmtrials_rows):,}"
)
display(invalid_mmtrials_rows)

## 1.8. Define the cleaning rules

After reviewing the dictionary and observed values, apply the following rules:

- preserve the original MMSE total and trial count;
- convert total scores outside 0--30 to missing;
- convert trial counts outside 1--6 to missing;
- treat `DONE = 0` as explicitly not completed;
- treat missing `DONE` in ADNI1, ADNIGO, and ADNI2 as structural absence;
- keep the one possible newer-phase record with missing `DONE` separate for review;
- represent administration source as `in_person`, `remote`, or `not_collected`;
- represent QC status as `passed`, `failed`, or `not_collected`;
- define a total score as usable only when it is valid, not explicitly uncompleted, and not associated with a documented QC failure;
- retain all rows for audit and perform no imputation.

In [ ]:
mmse_clean = mmse_raw.copy()

mmse_clean["VISDATE"] = pd.to_datetime(
    mmse_clean["VISDATE"],
    errors="coerce",
)

mmse_clean["MMSCORE_ORIGINAL"] = (
    mmse_clean["MMSCORE"]
)
mmse_clean["MMTRIALS_ORIGINAL"] = (
    mmse_clean["MMTRIALS"]
)

mmse_clean["MMSCORE_CLEAN"] = (
    mmse_clean["MMSCORE"]
    .where(mmse_clean["MMSCORE"].between(0, 30))
)

mmse_clean["MMTRIALS_CLEAN"] = (
    mmse_clean["MMTRIALS"]
    .where(mmse_clean["MMTRIALS"].between(1, 6))
)

older_phases = ["ADNI1", "ADNIGO", "ADNI2"]
newer_phases = ["ADNI3", "ADNI4"]

mmse_clean["MMSE_EXPLICITLY_NOT_DONE"] = (
    mmse_clean["DONE"].eq(0)
)

mmse_clean["MMSE_EXPLICITLY_DONE"] = (
    mmse_clean["DONE"].eq(1)
)

mmse_clean["MMSE_DONE_FIELD_NOT_COLLECTED"] = (
    mmse_clean["PHASE"].isin(older_phases)
    & mmse_clean["DONE"].isna()
)

mmse_clean["MMSE_NEWER_PHASE_DONE_MISSING"] = (
    mmse_clean["PHASE"].isin(newer_phases)
    & mmse_clean["DONE"].isna()
)

mmse_clean["MMSE_ADMINISTRATION_SOURCE"] = np.select(
    [
        mmse_clean["SOURCE"].eq(1),
        mmse_clean["SOURCE"].eq(2),
    ],
    [
        "in_person",
        "remote",
    ],
    default="not_collected",
)

mmse_clean["MMSE_QC_STATUS"] = np.select(
    [
        mmse_clean["HAS_QC_ERROR"].eq(0),
        mmse_clean["HAS_QC_ERROR"].eq(1),
    ],
    [
        "passed",
        "failed",
    ],
    default="not_collected",
)

mmse_clean["MMSE_SCORE_USABLE"] = (
    mmse_clean["MMSCORE_CLEAN"].notna()
    & ~mmse_clean["MMSE_EXPLICITLY_NOT_DONE"]
    & ~mmse_clean["MMSE_QC_STATUS"].eq("failed")
)

mmse_clean.loc[
    ~mmse_clean["MMSE_SCORE_USABLE"],
    "MMSCORE_CLEAN",
] = np.nan

cleaning_summary = pd.DataFrame(
    {
        "CHECK": [
            "Raw rows",
            "Invalid MMSCORE converted to missing",
            "Invalid MMTRIALS converted to missing",
            "Explicitly not-done assessments",
            "Newer-phase rows with missing DONE",
            "Remote assessments",
            "Documented QC failures",
            "Usable cleaned MMSE totals",
        ],
        "ROW_COUNT": [
            len(mmse_clean),
            int(
                (
                    mmse_clean["MMSCORE_ORIGINAL"].notna()
                    & ~mmse_clean["MMSCORE_ORIGINAL"].between(0, 30)
                ).sum()
            ),
            int(
                (
                    mmse_clean["MMTRIALS_ORIGINAL"].notna()
                    & ~mmse_clean["MMTRIALS_ORIGINAL"].between(1, 6)
                ).sum()
            ),
            int(mmse_clean["MMSE_EXPLICITLY_NOT_DONE"].sum()),
            int(mmse_clean["MMSE_NEWER_PHASE_DONE_MISSING"].sum()),
            int(
                mmse_clean["MMSE_ADMINISTRATION_SOURCE"]
                .eq("remote")
                .sum()
            ),
            int(mmse_clean["MMSE_QC_STATUS"].eq("failed").sum()),
            int(mmse_clean["MMSE_SCORE_USABLE"].sum()),
        ],
    }
)

display(cleaning_summary)

## 1.9. Review the newer-phase record with missing `DONE`

Because `DONE` is expected in ADNI3 and ADNI4, inspect every newer-phase row where it is missing before allowing the row to remain usable.

In [ ]:
newer_phase_missing_done = mmse_clean.loc[
    mmse_clean["MMSE_NEWER_PHASE_DONE_MISSING"],
    [
        "PHASE", "PTID", "RID",
        "VISCODE", "VISCODE2", "VISDATE",
        "DONE", "NDREASON", "SOURCE",
        "MMSCORE_ORIGINAL", "MMSCORE_CLEAN",
        "MMSE_SCORE_USABLE",
    ],
].copy()

print(
    "Newer-phase rows with missing DONE: "
    f"{len(newer_phase_missing_done):,}"
)
display(newer_phase_missing_done)

## 1.10. Derive harmonised MMSE domain scores

derive complete-case domain scores from the binary items:

- temporal orientation, 0--5;
- spatial orientation, 0--5;
- total orientation, 0--10;
- immediate registration, 0--3;
- attention/calculation, 0--5;
- delayed recall, 0--3;
- language and commands, 0--9.

For ADNI1, ADNIGO, and ADNI2, attention will be the sum of `MMD`, `MML`, `MMR`, `MMO`, and `MMW`. For ADNI3 and ADNI4, it will use `WORLDSCORE`.

Raw derived scores will be preserved for QC. Modelling versions will be set to missing whenever the MMSE assessment is not usable.

In [ ]:
temporal_orientation_items = [
    "MMDATE", "MMYEAR", "MMMONTH", "MMDAY", "MMSEASON",
]

spatial_orientation_items = [
    "MMHOSPIT", "MMFLOOR", "MMCITY", "MMAREA", "MMSTATE",
]

registration_items = [
    "WORD1", "WORD2", "WORD3",
]

older_attention_items = [
    "MMD", "MML", "MMR", "MMO", "MMW",
]

delayed_recall_items = [
    "WORD1DL", "WORD2DL", "WORD3DL",
]

language_command_items = [
    "MMWATCH", "MMPENCIL", "MMREPEAT",
    "MMHAND", "MMFOLD", "MMONFLR",
    "MMREAD", "MMWRITE", "MMDRAW",
]


def complete_item_sum(dataframe, columns):
    return dataframe[columns].sum(
        axis=1,
        min_count=len(columns),
    )


mmse_clean["MMSE_TEMPORAL_ORIENTATION_SCORE_RAW"] = (
    complete_item_sum(
        mmse_clean,
        temporal_orientation_items,
    )
)

mmse_clean["MMSE_SPATIAL_ORIENTATION_SCORE_RAW"] = (
    complete_item_sum(
        mmse_clean,
        spatial_orientation_items,
    )
)

mmse_clean["MMSE_ORIENTATION_SCORE_RAW"] = (
    mmse_clean[
        [
            "MMSE_TEMPORAL_ORIENTATION_SCORE_RAW",
            "MMSE_SPATIAL_ORIENTATION_SCORE_RAW",
        ]
    ].sum(axis=1, min_count=2)
)

mmse_clean["MMSE_REGISTRATION_SCORE_RAW"] = (
    complete_item_sum(
        mmse_clean,
        registration_items,
    )
)

mmse_clean["MMSE_ATTENTION_SCORE_OLDER_RAW"] = (
    complete_item_sum(
        mmse_clean,
        older_attention_items,
    )
)

mmse_clean["MMSE_ATTENTION_SCORE_RAW"] = np.where(
    mmse_clean["PHASE"].isin(older_phases),
    mmse_clean["MMSE_ATTENTION_SCORE_OLDER_RAW"],
    mmse_clean["WORLDSCORE"],
)

mmse_clean["MMSE_ATTENTION_SCORE_RAW"] = pd.to_numeric(
    mmse_clean["MMSE_ATTENTION_SCORE_RAW"],
    errors="coerce",
)

mmse_clean["MMSE_DELAYED_RECALL_SCORE_RAW"] = (
    complete_item_sum(
        mmse_clean,
        delayed_recall_items,
    )
)

mmse_clean["MMSE_LANGUAGE_COMMAND_SCORE_RAW"] = (
    complete_item_sum(
        mmse_clean,
        language_command_items,
    )
)

raw_to_model_domain_columns = {
    "MMSE_TEMPORAL_ORIENTATION_SCORE_RAW":
        "MMSE_TEMPORAL_ORIENTATION_SCORE",
    "MMSE_SPATIAL_ORIENTATION_SCORE_RAW":
        "MMSE_SPATIAL_ORIENTATION_SCORE",
    "MMSE_ORIENTATION_SCORE_RAW":
        "MMSE_ORIENTATION_SCORE",
    "MMSE_REGISTRATION_SCORE_RAW":
        "MMSE_REGISTRATION_SCORE",
    "MMSE_ATTENTION_SCORE_RAW":
        "MMSE_ATTENTION_SCORE",
    "MMSE_DELAYED_RECALL_SCORE_RAW":
        "MMSE_DELAYED_RECALL_SCORE",
    "MMSE_LANGUAGE_COMMAND_SCORE_RAW":
        "MMSE_LANGUAGE_COMMAND_SCORE",
}

for raw_column, model_column in raw_to_model_domain_columns.items():
    mmse_clean[model_column] = (
        mmse_clean[raw_column]
        .where(mmse_clean["MMSE_SCORE_USABLE"])
    )

domain_ranges = {
    "MMSE_TEMPORAL_ORIENTATION_SCORE": (0, 5),
    "MMSE_SPATIAL_ORIENTATION_SCORE": (0, 5),
    "MMSE_ORIENTATION_SCORE": (0, 10),
    "MMSE_REGISTRATION_SCORE": (0, 3),
    "MMSE_ATTENTION_SCORE": (0, 5),
    "MMSE_DELAYED_RECALL_SCORE": (0, 3),
    "MMSE_LANGUAGE_COMMAND_SCORE": (0, 9),
}

domain_range_audit = []

for column, (minimum, maximum) in domain_ranges.items():
    invalid_mask = (
        mmse_clean[column].notna()
        & ~mmse_clean[column].between(minimum, maximum)
    )
    domain_range_audit.append(
        {
            "DOMAIN_SCORE": column,
            "NON_MISSING": int(
                mmse_clean[column].notna().sum()
            ),
            "EXPECTED_MIN": minimum,
            "EXPECTED_MAX": maximum,
            "INVALID_ROW_COUNT": int(invalid_mask.sum()),
        }
    )

domain_range_audit = pd.DataFrame(domain_range_audit)
display(domain_range_audit)

## 1.11. Reconstruct and compare the total score

reconstruct a 30-point total only when every domain is complete. The reconstructed total is a QC check and will never replace the official ADNI `MMSCORE`.

retain:

- the reconstructed total;
- the difference between reconstructed and reported totals;
- an exact-match flag;
- a domain-total consistency flag.

The domain scores remain optional secondary features. The official total is the primary MMSE feature.

In [ ]:
domain_columns = [
    "MMSE_ORIENTATION_SCORE",
    "MMSE_REGISTRATION_SCORE",
    "MMSE_ATTENTION_SCORE",
    "MMSE_DELAYED_RECALL_SCORE",
    "MMSE_LANGUAGE_COMMAND_SCORE",
]

mmse_clean["MMSE_RECONSTRUCTED_TOTAL"] = (
    mmse_clean[domain_columns]
    .sum(axis=1, min_count=len(domain_columns))
)

mmse_clean["MMSE_TOTAL_DIFFERENCE"] = (
    mmse_clean["MMSE_RECONSTRUCTED_TOTAL"]
    - mmse_clean["MMSCORE_CLEAN"]
)

comparison_mask = (
    mmse_clean["MMSE_RECONSTRUCTED_TOTAL"].notna()
    & mmse_clean["MMSCORE_CLEAN"].notna()
)

mmse_clean["MMSE_TOTAL_EXACT_MATCH"] = (
    comparison_mask
    & mmse_clean["MMSE_TOTAL_DIFFERENCE"].eq(0)
)

mmse_clean["MMSE_DOMAIN_TOTAL_CONSISTENT"] = np.select(
    [
        ~comparison_mask,
        mmse_clean["MMSE_TOTAL_DIFFERENCE"].eq(0),
    ],
    [
        "not_comparable",
        "exact_match",
    ],
    default="mismatch",
)

total_comparison_summary = pd.DataFrame(
    {
        "CHECK": [
            "Rows with reported and reconstructed totals",
            "Exact matches",
            "Mismatches",
        ],
        "ROW_COUNT": [
            int(comparison_mask.sum()),
            int(mmse_clean["MMSE_TOTAL_EXACT_MATCH"].sum()),
            int(
                mmse_clean["MMSE_DOMAIN_TOTAL_CONSISTENT"]
                .eq("mismatch")
                .sum()
            ),
        ],
    }
)

display(total_comparison_summary)

print("\nDifference distribution:")
display(
    mmse_clean.loc[
        comparison_mask,
        "MMSE_TOTAL_DIFFERENCE",
    ]
    .value_counts()
    .sort_index()
    .rename_axis("DIFFERENCE")
    .reset_index(name="ROW_COUNT")
)

## 1.12. Inspect reconstructed-total mismatches

inspect all mismatches and separately identify large discrepancies. These records will not cause the official total to be changed, but they will be excluded from any domain-score modelling analysis unless reviewed deliberately.

In [ ]:
mmse_total_mismatches = mmse_clean.loc[
    mmse_clean["MMSE_DOMAIN_TOTAL_CONSISTENT"].eq("mismatch"),
    [
        "PHASE", "PTID", "RID",
        "VISCODE", "VISCODE2", "VISDATE",
        "MMSCORE_CLEAN",
        "MMSE_RECONSTRUCTED_TOTAL",
        "MMSE_TOTAL_DIFFERENCE",
        *domain_columns,
    ],
].copy()

mmse_large_total_mismatches = mmse_total_mismatches.loc[
    mmse_total_mismatches["MMSE_TOTAL_DIFFERENCE"]
    .abs()
    .ge(3)
].copy()

print(
    "All reconstructed-total mismatches: "
    f"{len(mmse_total_mismatches):,}"
)
print(
    "Large absolute differences (>= 3 points): "
    f"{len(mmse_large_total_mismatches):,}"
)

display(mmse_large_total_mismatches)

## 1.13. Standardise identifiers, visit codes, and temporal eligibility

retain `RID` as the primary participant identifier and use `VISCODE2` as the preferred harmonised visit code, falling back to `VISCODE` when necessary.

Rows without an assessment date will remain in the full audit table, but they will be marked as temporally ineligible because they cannot later be aligned to the common reference date.

In [ ]:
mmse_clean["RID"] = pd.to_numeric(
    mmse_clean["RID"],
    errors="coerce",
).astype("Int64")

for column in ["PHASE", "PTID", "VISCODE", "VISCODE2"]:
    mmse_clean[column] = (
        mmse_clean[column]
        .astype("string")
        .str.strip()
        .replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "<NA>": pd.NA,
            }
        )
    )

mmse_clean["MMSE_VISIT_CODE"] = (
    mmse_clean["VISCODE2"]
    .fillna(mmse_clean["VISCODE"])
)

mmse_clean["MMSE_TEMPORALLY_ELIGIBLE"] = (
    mmse_clean["RID"].notna()
    & mmse_clean["VISDATE"].notna()
)

mmse_clean["MMSE_RECORD_KEY"] = (
    mmse_clean["RID"].astype("string")
    + "_"
    + mmse_clean["PHASE"].fillna("unknown-phase")
    + "_"
    + mmse_clean["MMSE_VISIT_CODE"].fillna("unknown-visit")
    + "_"
    + mmse_clean["VISDATE"]
    .dt.strftime("%Y-%m-%d")
    .fillna("unknown-date")
)

temporal_summary = pd.DataFrame(
    {
        "CHECK": [
            "Rows missing RID",
            "Rows missing VISDATE",
            "Rows temporally eligible",
        ],
        "ROW_COUNT": [
            int(mmse_clean["RID"].isna().sum()),
            int(mmse_clean["VISDATE"].isna().sum()),
            int(mmse_clean["MMSE_TEMPORALLY_ELIGIBLE"].sum()),
        ],
    }
)

display(temporal_summary)

## 1.14. Audit exact and near-duplicate assessment records

Repeated longitudinal rows are not automatically errors. distinguish:

- exact duplicate rows;
- repeated participant and assessment-date combinations;
- repeated participant and harmonised-visit combinations;
- repeated participant, date, and visit combinations;
- conflicting cleaned totals within any repeated group.

inspect the repeated-date and repeated-visit records rather than silently collapse them.

In [ ]:
mmse_clean["MMSE_EXACT_DUPLICATE_ROW"] = (
    mmse_raw.duplicated(keep=False)
)

mmse_clean["MMSE_REPEATED_RID_DATE"] = (
    mmse_clean.duplicated(
        subset=["RID", "VISDATE"],
        keep=False,
    )
    & mmse_clean["RID"].notna()
    & mmse_clean["VISDATE"].notna()
)

mmse_clean["MMSE_REPEATED_RID_VISIT"] = (
    mmse_clean.duplicated(
        subset=["RID", "MMSE_VISIT_CODE"],
        keep=False,
    )
    & mmse_clean["RID"].notna()
    & mmse_clean["MMSE_VISIT_CODE"].notna()
)

strict_key = [
    "RID",
    "VISDATE",
    "MMSE_VISIT_CODE",
]

mmse_clean["MMSE_REPEATED_STRICT_KEY"] = (
    mmse_clean.duplicated(
        subset=strict_key,
        keep=False,
    )
    & mmse_clean["RID"].notna()
    & mmse_clean["VISDATE"].notna()
    & mmse_clean["MMSE_VISIT_CODE"].notna()
)

def add_conflict_flag(dataframe, group_columns, flag_name):
    counts = (
        dataframe.loc[
            dataframe.duplicated(
                subset=group_columns,
                keep=False,
            )
        ]
        .groupby(group_columns, dropna=False)["MMSCORE_CLEAN"]
        .nunique(dropna=True)
        .rename("_DISTINCT_TOTALS")
        .reset_index()
    )
    result = dataframe.merge(
        counts,
        on=group_columns,
        how="left",
    )
    result[flag_name] = (
        result["_DISTINCT_TOTALS"]
        .fillna(0)
        .gt(1)
    )
    return result.drop(columns="_DISTINCT_TOTALS")

mmse_clean = add_conflict_flag(
    mmse_clean,
    ["RID", "VISDATE"],
    "MMSE_CONFLICTING_RID_DATE_SCORE",
)

mmse_clean = add_conflict_flag(
    mmse_clean,
    ["RID", "MMSE_VISIT_CODE"],
    "MMSE_CONFLICTING_RID_VISIT_SCORE",
)

mmse_clean = add_conflict_flag(
    mmse_clean,
    strict_key,
    "MMSE_CONFLICTING_STRICT_KEY_SCORE",
)

duplicate_summary = pd.DataFrame(
    {
        "CHECK": [
            "Exact duplicate rows",
            "Rows in repeated RID-date groups",
            "Rows in repeated RID-visit groups",
            "Rows in repeated strict-key groups",
            "Rows in conflicting RID-date groups",
            "Rows in conflicting RID-visit groups",
            "Rows in conflicting strict-key groups",
        ],
        "ROW_COUNT": [
            int(mmse_clean["MMSE_EXACT_DUPLICATE_ROW"].sum()),
            int(mmse_clean["MMSE_REPEATED_RID_DATE"].sum()),
            int(mmse_clean["MMSE_REPEATED_RID_VISIT"].sum()),
            int(mmse_clean["MMSE_REPEATED_STRICT_KEY"].sum()),
            int(
                mmse_clean["MMSE_CONFLICTING_RID_DATE_SCORE"]
                .sum()
            ),
            int(
                mmse_clean["MMSE_CONFLICTING_RID_VISIT_SCORE"]
                .sum()
            ),
            int(
                mmse_clean["MMSE_CONFLICTING_STRICT_KEY_SCORE"]
                .sum()
            ),
        ],
    }
)

display(duplicate_summary)

## 1.15. Review repeated-date and repeated-visit records

display every record involved in a repeated participant-date or participant-visit group. No record will be removed automatically unless an exact duplicate or a truly duplicated strict assessment key is found and reviewed.

In [ ]:
duplicate_review_columns = [
    "PHASE", "PTID", "RID",
    "VISCODE", "VISCODE2", "MMSE_VISIT_CODE",
    "VISDATE", "ID",
    "MMSCORE_ORIGINAL", "MMSCORE_CLEAN",
    "MMSE_SCORE_USABLE",
    "MMSE_ADMINISTRATION_SOURCE",
    "MMSE_REPEATED_RID_DATE",
    "MMSE_REPEATED_RID_VISIT",
    "MMSE_REPEATED_STRICT_KEY",
    "MMSE_CONFLICTING_RID_DATE_SCORE",
    "MMSE_CONFLICTING_RID_VISIT_SCORE",
]

mmse_repeated_records_review = (
    mmse_clean.loc[
        mmse_clean["MMSE_REPEATED_RID_DATE"]
        | mmse_clean["MMSE_REPEATED_RID_VISIT"],
        duplicate_review_columns,
    ]
    .sort_values(
        ["RID", "VISDATE", "MMSE_VISIT_CODE"],
        kind="stable",
    )
)

display(mmse_repeated_records_review)

## 1.16. Decide whether any records should be collapsed

collapse records only when the strict key (`RID`, assessment date, and harmonised visit code) is repeated. If no strict-key duplicates exist, the cleaned longitudinal table will preserve all rows.

This avoids forcing potentially legitimate repeated-date or repeated-visit records into a single row.

In [ ]:
strict_duplicate_rows = int(
    mmse_clean["MMSE_REPEATED_STRICT_KEY"].sum()
)

if strict_duplicate_rows == 0:
    mmse_longitudinal_clean = (
        mmse_clean
        .sort_values(
            ["RID", "VISDATE", "MMSE_VISIT_CODE"],
            kind="stable",
        )
        .reset_index(drop=True)
    )
    print(
        "No repeated strict assessment keys were found. "
        "No rows were collapsed."
    )
else:
    raise ValueError(
        "Repeated strict assessment keys were found. "
        "Review them before selecting preferred records."
    )

print(
    "Rows in cleaned longitudinal table: "
    f"{len(mmse_longitudinal_clean):,}"
)

## 1.17. Create separate model-ready and QC outputs

keep modelling features separate from validation and administrative metadata.

The primary model-ready table will contain:

- identifiers and visit/date fields;
- the cleaned MMSE total score;
- temporal eligibility;
- score usability.

A separate optional domain-feature table will contain the harmonised domain scores. Domain scores will only be present for usable assessments and will carry the reconstructed-total consistency status.

The QC table will contain completion, source, QC, reconstruction, and duplicate-review fields.

In [ ]:
primary_feature_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "MMSE_VISIT_CODE",
    "VISDATE",
    "MMSCORE_CLEAN",
    "MMSE_SCORE_USABLE",
    "MMSE_TEMPORALLY_ELIGIBLE",
    "MMSE_RECORD_KEY",
]

mmse_primary_features_v2 = (
    mmse_longitudinal_clean[primary_feature_columns]
    .rename(
        columns={
            "MMSCORE_CLEAN": "MMSE_TOTAL_SCORE",
        }
    )
    .copy()
)

domain_feature_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "MMSE_VISIT_CODE",
    "VISDATE",
    "MMSCORE_CLEAN",
    "MMSE_ORIENTATION_SCORE",
    "MMSE_REGISTRATION_SCORE",
    "MMSE_ATTENTION_SCORE",
    "MMSE_DELAYED_RECALL_SCORE",
    "MMSE_LANGUAGE_COMMAND_SCORE",
    "MMSE_DOMAIN_TOTAL_CONSISTENT",
    "MMSE_SCORE_USABLE",
    "MMSE_TEMPORALLY_ELIGIBLE",
    "MMSE_RECORD_KEY",
]

mmse_optional_domain_features_v2 = (
    mmse_longitudinal_clean[domain_feature_columns]
    .rename(
        columns={
            "MMSCORE_CLEAN": "MMSE_TOTAL_SCORE",
        }
    )
    .copy()
)

qc_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "MMSE_VISIT_CODE",
    "VISDATE",
    "DONE",
    "NDREASON",
    "SOURCE",
    "MMSE_ADMINISTRATION_SOURCE",
    "MMSE_QC_STATUS",
    "MMSCORE_ORIGINAL",
    "MMSCORE_CLEAN",
    "MMTRIALS_ORIGINAL",
    "MMTRIALS_CLEAN",
    "MMSE_EXPLICITLY_NOT_DONE",
    "MMSE_NEWER_PHASE_DONE_MISSING",
    "MMSE_SCORE_USABLE",
    "MMSE_TEMPORALLY_ELIGIBLE",
    "MMSE_RECONSTRUCTED_TOTAL",
    "MMSE_TOTAL_DIFFERENCE",
    "MMSE_DOMAIN_TOTAL_CONSISTENT",
    "MMSE_REPEATED_RID_DATE",
    "MMSE_REPEATED_RID_VISIT",
    "MMSE_REPEATED_STRICT_KEY",
    "MMSE_CONFLICTING_RID_DATE_SCORE",
    "MMSE_CONFLICTING_RID_VISIT_SCORE",
    "MMSE_CONFLICTING_STRICT_KEY_SCORE",
    "MMSE_RECORD_KEY",
]

mmse_qc_records_v2 = (
    mmse_longitudinal_clean[qc_columns]
    .copy()
)

print("Primary feature table:")
print(f"Rows: {len(mmse_primary_features_v2):,}")
print(
    "Usable totals: "
    f"{mmse_primary_features_v2['MMSE_TOTAL_SCORE'].notna().sum():,}"
)

print("\nOptional domain-feature table:")
print(f"Rows: {len(mmse_optional_domain_features_v2):,}")
print(
    "Rows with all five domain scores: "
    f"{mmse_optional_domain_features_v2[domain_columns].notna().all(axis=1).sum():,}"
)

display(mmse_primary_features_v2.head())

## 1.18. Produce the final QC summary

summarise the complete preprocessing result, including participant coverage, cleaning actions, temporal eligibility, domain-score consistency, and duplicate-review findings.

In [ ]:
mmse_qc_summary_v2 = pd.DataFrame(
    [
        {
            "QC_METRIC": "Raw rows",
            "VALUE": len(mmse_raw),
        },
        {
            "QC_METRIC": "Raw unique participants",
            "VALUE": mmse_raw["RID"].nunique(dropna=True),
        },
        {
            "QC_METRIC": "Invalid MMSCORE values converted to missing",
            "VALUE": int(
                (
                    mmse_longitudinal_clean["MMSCORE_ORIGINAL"].notna()
                    & ~mmse_longitudinal_clean[
                        "MMSCORE_ORIGINAL"
                    ].between(0, 30)
                ).sum()
            ),
        },
        {
            "QC_METRIC": "Invalid MMTRIALS values converted to missing",
            "VALUE": int(
                (
                    mmse_longitudinal_clean["MMTRIALS_ORIGINAL"].notna()
                    & ~mmse_longitudinal_clean[
                        "MMTRIALS_ORIGINAL"
                    ].between(1, 6)
                ).sum()
            ),
        },
        {
            "QC_METRIC": "Explicitly not-done assessments",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_EXPLICITLY_NOT_DONE"
                ].sum()
            ),
        },
        {
            "QC_METRIC": "Newer-phase rows with missing DONE",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_NEWER_PHASE_DONE_MISSING"
                ].sum()
            ),
        },
        {
            "QC_METRIC": "Remote assessments",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_ADMINISTRATION_SOURCE"
                ].eq("remote").sum()
            ),
        },
        {
            "QC_METRIC": "Documented QC failures",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_QC_STATUS"
                ].eq("failed").sum()
            ),
        },
        {
            "QC_METRIC": "Rows with usable MMSE total",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_SCORE_USABLE"
                ].sum()
            ),
        },
        {
            "QC_METRIC": "Participants with at least one usable total",
            "VALUE": int(
                mmse_longitudinal_clean.loc[
                    mmse_longitudinal_clean[
                        "MMSE_SCORE_USABLE"
                    ],
                    "RID",
                ].nunique(dropna=True)
            ),
        },
        {
            "QC_METRIC": "Rows missing assessment date",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "VISDATE"
                ].isna().sum()
            ),
        },
        {
            "QC_METRIC": "Rows temporally eligible",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_TEMPORALLY_ELIGIBLE"
                ].sum()
            ),
        },
        {
            "QC_METRIC": "Rows with reconstructable total",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_RECONSTRUCTED_TOTAL"
                ].notna().sum()
            ),
        },
        {
            "QC_METRIC": "Exact reconstructed-total matches",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_DOMAIN_TOTAL_CONSISTENT"
                ].eq("exact_match").sum()
            ),
        },
        {
            "QC_METRIC": "Reconstructed-total mismatches",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_DOMAIN_TOTAL_CONSISTENT"
                ].eq("mismatch").sum()
            ),
        },
        {
            "QC_METRIC": "Rows in repeated RID-date groups",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_REPEATED_RID_DATE"
                ].sum()
            ),
        },
        {
            "QC_METRIC": "Rows in repeated RID-visit groups",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_REPEATED_RID_VISIT"
                ].sum()
            ),
        },
        {
            "QC_METRIC": "Rows in repeated strict-key groups",
            "VALUE": int(
                mmse_longitudinal_clean[
                    "MMSE_REPEATED_STRICT_KEY"
                ].sum()
            ),
        },
        {
            "QC_METRIC": "Final primary feature rows",
            "VALUE": len(mmse_primary_features_v2),
        },
    ]
)

display(mmse_qc_summary_v2)

## 1.19. Save versioned MMSE outputs

save versioned outputs so that this corrected notebook does not overwrite the earlier MMSE files.

The outputs are:

1. full cleaned longitudinal audit table;
2. primary longitudinal MMSE feature table;
3. optional MMSE domain-feature table;
4. row-level QC table;
5. QC summary;
6. dictionary definitions and coverage;
7. phase-missingness audit;
8. domain-range audit;
9. reconstructed-total mismatches;
10. large reconstructed-total mismatches;
11. repeated-record review;
12. newer-phase missing-`DONE` review.

The raw MMSE source remains unchanged.

In [ ]:
output_paths = {
    "full_cleaned_audit": (
        MMSE_INTERIM_DIR
        / "mmse_longitudinal_full_cleaned_audit_v2.csv"
    ),
    "primary_features": (
        MMSE_PROCESSED_DIR
        / "mmse_longitudinal_primary_features_v2.csv"
    ),
    "optional_domain_features": (
        MMSE_PROCESSED_DIR
        / "mmse_longitudinal_optional_domain_features_v2.csv"
    ),
    "row_level_qc": (
        MMSE_QC_DIR
        / "mmse_row_level_qc_v2.csv"
    ),
    "qc_summary": (
        MMSE_QC_DIR
        / "mmse_preprocessing_qc_summary_v2.csv"
    ),
    "datadic_definitions": (
        MMSE_QC_DIR
        / "mmse_datadic_phase_specific_definitions_v2.csv"
    ),
    "datadic_coverage": (
        MMSE_QC_DIR
        / "mmse_datadic_coverage_v2.csv"
    ),
    "phase_missingness": (
        MMSE_QC_DIR
        / "mmse_phase_missingness_v2.csv"
    ),
    "domain_range_audit": (
        MMSE_QC_DIR
        / "mmse_domain_range_audit_v2.csv"
    ),
    "total_mismatches": (
        MMSE_QC_DIR
        / "mmse_total_reconstruction_mismatches_v2.csv"
    ),
    "large_total_mismatches": (
        MMSE_QC_DIR
        / "mmse_large_total_reconstruction_mismatches_v2.csv"
    ),
    "repeated_records_review": (
        MMSE_QC_DIR
        / "mmse_repeated_records_review_v2.csv"
    ),
    "newer_phase_missing_done": (
        MMSE_QC_DIR
        / "mmse_newer_phase_missing_done_review_v2.csv"
    ),
}

mmse_longitudinal_clean.to_csv(
    output_paths["full_cleaned_audit"],
    index=False,
)

mmse_primary_features_v2.to_csv(
    output_paths["primary_features"],
    index=False,
)

mmse_optional_domain_features_v2.to_csv(
    output_paths["optional_domain_features"],
    index=False,
)

mmse_qc_records_v2.to_csv(
    output_paths["row_level_qc"],
    index=False,
)

mmse_qc_summary_v2.to_csv(
    output_paths["qc_summary"],
    index=False,
)

mmse_datadic_relevant.to_csv(
    output_paths["datadic_definitions"],
    index=False,
)

dictionary_coverage.to_csv(
    output_paths["datadic_coverage"],
    index=False,
)

phase_missingness.to_csv(
    output_paths["phase_missingness"],
    index=False,
)

domain_range_audit.to_csv(
    output_paths["domain_range_audit"],
    index=False,
)

mmse_total_mismatches.to_csv(
    output_paths["total_mismatches"],
    index=False,
)

mmse_large_total_mismatches.to_csv(
    output_paths["large_total_mismatches"],
    index=False,
)

mmse_repeated_records_review.to_csv(
    output_paths["repeated_records_review"],
    index=False,
)

newer_phase_missing_done.to_csv(
    output_paths["newer_phase_missing_done"],
    index=False,
)

print("Versioned MMSE outputs saved successfully.\n")

for name, path in output_paths.items():
    print(f"{name}:\n{path}\n")

## 1.20. Final validation

reload the three principal outputs from disk and verify that:

- their row counts match the in-memory tables;
- the primary feature table contains no invalid MMSE totals;
- unusable assessments do not contain model-ready domain scores;
- versioned files were created successfully.

In [ ]:
reloaded_full = pd.read_csv(
    output_paths["full_cleaned_audit"],
    low_memory=False,
)

reloaded_primary = pd.read_csv(
    output_paths["primary_features"],
    low_memory=False,
)

reloaded_domains = pd.read_csv(
    output_paths["optional_domain_features"],
    low_memory=False,
)

assert len(reloaded_full) == len(mmse_longitudinal_clean)
assert len(reloaded_primary) == len(mmse_primary_features_v2)
assert len(reloaded_domains) == len(mmse_optional_domain_features_v2)

assert not (
    reloaded_primary["MMSE_TOTAL_SCORE"].notna()
    & ~reloaded_primary["MMSE_TOTAL_SCORE"].between(0, 30)
).any()

unusable_domain_rows = (
    ~reloaded_domains["MMSE_SCORE_USABLE"].astype(bool)
)

assert not reloaded_domains.loc[
    unusable_domain_rows,
    [
        "MMSE_ORIENTATION_SCORE",
        "MMSE_REGISTRATION_SCORE",
        "MMSE_ATTENTION_SCORE",
        "MMSE_DELAYED_RECALL_SCORE",
        "MMSE_LANGUAGE_COMMAND_SCORE",
    ],
].notna().any().any()

for path in output_paths.values():
    assert path.exists()

print("All final validation checks passed.")
print(
    "The corrected v2 outputs are internally consistent "
    "and the earlier MMSE outputs were not overwritten."
)